# Huấn luyện MNIST bằng MPI trên 1 GPU

Luồng: chia dữ liệu → forward → backward → gom gradient vào một bộ đệm → GPU sang RAM → MPI Allreduce → RAM về GPU → SGD.

Dùng 1 tiến trình MPI và 1 GPU. Allreduce vẫn được gọi nhưng không trao đổi gradient với tiến trình khác.


In [1]:
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "Hãy bật GPU tại Runtime > Change runtime type."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "openmpi-bin", "libopenmpi-dev"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mpi4py"], check=True)
subprocess.run(["mpirun", "--version"], check=True)


GPU: Tesla T4
PyTorch: 2.14.1+cu130


CompletedProcess(args=['mpirun', '--version'], returncode=0)

## Tải MNIST một lần

Huấn luyện dùng 60.000 ảnh; đánh giá cuối dùng 10.000 ảnh test. File dữ liệu nằm trên máy ảo Colab.


In [2]:
from torchvision.datasets import MNIST

MNIST("./data", train=True, download=True)
MNIST("./data", train=False, download=True)
print("Đã tải MNIST.")


100%|██████████| 9.91M/9.91M [00:00<00:00, 14.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 448kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.14MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.77MB/s]

Đã tải MNIST.


## Tạo chương trình MPI

MPI chạy file Python riêng. Hàm `average_gradients()` là phần cần chú ý: `Allreduce(SUM)` rồi chia số tiến trình. Với một tiến trình, gradient trước và sau bước này có cùng giá trị. Bước `optimizer.step()` luôn thực hiện, kể cả khi chỉ có một GPU.


In [3]:
%%writefile train_mpi.py
"""MNIST + PyTorch + MPI; start with one MPI rank and one GPU."""

import argparse

import numpy as np
import torch
from mpi4py import MPI
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms


class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(1, 32, 3), nn.ReLU(),       # 28 -> 26
            nn.Conv2d(32, 64, 3), nn.ReLU(),     # 26 -> 24
            nn.MaxPool2d(2), nn.Flatten(),       # 64 x 12 x 12
            nn.Linear(9216, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.layers(x)


@torch.no_grad()
def average_gradients(model, comm, device):
    # Mot bucket cho toan bo gradient FP32: GPU -> RAM.
    params = list(model.parameters())
    flat = torch.cat([p.grad.reshape(-1) for p in params]).cpu().numpy()

    # Cong gradient cua moi rank; ket qua co mat tai TAT CA cac rank.
    comm.Allreduce(MPI.IN_PLACE, flat, op=MPI.SUM)
    flat /= comm.Get_size()

    # RAM -> GPU, sau do khoi phuc gradient cua tung tham so.
    averaged = torch.from_numpy(flat).to(device)
    offset = 0
    for p in params:
        count = p.numel()
        p.grad.copy_(averaged[offset:offset + count].view_as(p))
        offset += count


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--epochs', type=int, default=3)
    parser.add_argument('--batch-size', type=int, default=128)
    parser.add_argument('--data-dir', default='./data')
    args = parser.parse_args()
    if args.epochs < 1 or args.batch_size < 1:
        parser.error('epochs va batch-size phai lon hon 0')

    # 1. Khoi tao MPI va anh xa: moi local rank dung mot GPU.
    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()
    local = comm.Split_type(MPI.COMM_TYPE_SHARED)
    local_rank, local_size = local.Get_rank(), local.Get_size()
    local.Free()
    if not torch.cuda.is_available():
        raise RuntimeError('Hay bat GPU trong Runtime > Change runtime type.')
    if local_size > torch.cuda.device_count():
        raise RuntimeError('So tien trinh tren may vuot so GPU. Tren Colab hay dung -np 1.')
    torch.cuda.set_device(local_rank)
    device = torch.device('cuda', local_rank)
    torch.set_num_threads(1)
    torch.manual_seed(42)

    # 2. MNIST da duoc tai o cell chuan bi; cac rank cung doc thu muc nay.
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    train_data = datasets.MNIST(args.data_dir, train=True, download=False,
                                transform=transform)
    sampler = DistributedSampler(train_data, num_replicas=size, rank=rank,
                                  shuffle=True, seed=42, drop_last=True)
    # Sampler giu so mau/so batch bang nhau giua cac rank.
    # Voi size=1, dung du 60.000 mau; size>1 bo toi da size-1 mau/epoch.
    loader = DataLoader(train_data, batch_size=args.batch_size, sampler=sampler,
                        num_workers=0, pin_memory=True, drop_last=False)

    # 3. Khoi tao model; Bcast dam bao moi rank bat dau cung trong so.
    model = MNISTNet().to(device)
    with torch.no_grad():
        for p in model.parameters():
            initial = p.detach().cpu().numpy().copy()
            comm.Bcast(initial, root=0)
            p.copy_(torch.from_numpy(initial).to(device))
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    print(f'Rank {rank}/{size} | {torch.cuda.get_device_name(device)} | '
          f'local batch={args.batch_size} | global batch={args.batch_size * size}',
          flush=True)

    # 4. Vong lap huan luyen: forward -> backward -> Allreduce -> step.
    for epoch in range(args.epochs):
        sampler.set_epoch(epoch)
        model.train()
        loss_sum = torch.zeros((), device=device)
        seen = 0
        torch.cuda.synchronize(device)
        comm.Barrier()
        start = MPI.Wtime()

        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)                 # Forward
            loss = criterion(logits, labels)        # Loss
            loss.backward()                        # Gradient cuc bo
            average_gradients(model, comm, device) # Gradient trung binh
            optimizer.step()                       # Cap nhat trong so
            loss_sum += loss.detach() * labels.size(0)
            seen += labels.size(0)

        torch.cuda.synchronize(device)
        elapsed = MPI.Wtime() - start
        epoch_seconds = comm.allreduce(elapsed, op=MPI.MAX)
        stats = np.array([loss_sum.item(), seen], dtype=np.float64)
        comm.Allreduce(MPI.IN_PLACE, stats, op=MPI.SUM)
        if rank == 0:
            print(f'Epoch {epoch + 1}/{args.epochs} | '
                  f'loss={stats[0] / stats[1]:.4f} | '
                  f'train_time={epoch_seconds:.2f}s | '
                  f'samples/s={stats[1] / epoch_seconds:.0f}', flush=True)

    # 5. Rank 0 danh gia 10.000 anh test va luu model.
    if rank == 0:
        test_data = datasets.MNIST(args.data_dir, train=False, download=False,
                                   transform=transform)
        test_loader = DataLoader(test_data, batch_size=512, num_workers=0)
        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in test_loader:
                pred = model(images.to(device)).argmax(dim=1)
                correct += (pred.cpu() == labels).sum().item()
        print(f'Test accuracy: {100 * correct / len(test_data):.2f}%', flush=True)
        torch.save(model.state_dict(), 'mnist_mpi.pt')
        print('Da luu mnist_mpi.pt', flush=True)
    comm.Barrier()


if __name__ == '__main__':
    main()


Writing train_mpi.py


## Chạy 1 tiến trình trên 1 GPU

`-np 1` là số tiến trình. `--allow-run-as-root` dành cho môi trường máy ảo Colab chạy bằng root. Chạy thông qua `python -m mpi4py` để MPI hủy cả job nếu có ngoại lệ chưa được xử lý.


In [5]:
import subprocess
import sys

command = [
    "mpirun", "--allow-run-as-root", "--bind-to", "none",
    "-np", "1",
    sys.executable, "-u", "-m", "mpi4py", "train_mpi.py",
    "--epochs", "3",
    "--batch-size", "128",
]

with subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)

    returncode = process.wait()

if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)

print("Huấn luyện đã kết thúc.")

Rank 0/1 | Tesla T4 | local batch=128 | global batch=128
Epoch 1/3 | loss=0.3035 | train_time=17.08s | samples/s=3514
Epoch 2/3 | loss=0.0706 | train_time=16.44s | samples/s=3649
Epoch 3/3 | loss=0.0441 | train_time=16.56s | samples/s=3623
Test accuracy: 98.61%
Da luu mnist_mpi.pt
Huấn luyện đã kết thúc.


## Đọc kết quả

- `Rank 0/1`: chỉ một tiến trình; local batch và global batch cùng bằng 128 (batch cuối nhỏ hơn).
- `loss`: loss huấn luyện trung bình theo số ảnh trong epoch.
- `train_time`: thời gian vòng huấn luyện của epoch, gồm nạp batch, tính toán, sao chép gradient và MPI; không gồm tải MNIST, khởi tạo, đánh giá và lưu file.
- `samples/s`: số ảnh huấn luyện xử lý mỗi giây.
- `Test accuracy`: tỷ lệ đúng trên 10.000 ảnh test. Giá trị thực tế phụ thuộc lần chạy và môi trường.
- `mnist_mpi.pt`: trọng số mô hình đã lưu trong thư mục làm việc của Colab.


